In [0]:
CREATE WIDGET TEXT academic_year DEFAULT "202526";
CREATE WIDGET TEXT ilr_snapshot DEFAULT "SN06";
CREATE WIDGET TEXT snapshot DEFAULT "06";
CREATE WIDGET TEXT source_catalog DEFAULT "catalog_40_copper_proj_fe_skills_statistics_dev";    
CREATE WIDGET TEXT output_catalog DEFAULT "catalog_40_copper_proj_fe_skills_statistics_dev";
CREATE WIDGET TEXT source_schema DEFAULT "fe_skills_dev";                                       
CREATE WIDGET TEXT output_schema DEFAULT "output_202526_SN06"

In [0]:
--Select and define fields, and join on routes data

CREATE OR REPLACE TEMPORARY VIEW APPS AS

SELECT 
  CASE 
    WHEN a.academic_year = ${academic_year} THEN
      CASE 
        WHEN ${snapshot} = 4 THEN CONCAT(a.academic_year, ' (Aug to Oct)')
        WHEN ${snapshot} = 6 THEN CONCAT(a.academic_year, ' (Aug to Jan)')
        WHEN ${snapshot} = 10 THEN CONCAT(a.academic_year, ' (Aug to Apr)')
        ELSE a.academic_year
      END
    ELSE a.academic_year
  END AS `year`,
a.age_summary as age_group,

CASE WHEN a.age_summary = 'Under 19' then 1 WHEN a.age_summary = '19-24' then 2 WHEN a.age_summary = '25+' then 3 else 4 END as age_group_order,
a.apps_level,
CASE WHEN a.apps_level = 'Intermediate Apprenticeship' then 1 WHEN a.apps_level = 'Advanced Apprenticeship' then 2 WHEN a.apps_level = 'Higher Apprenticeship' then 3 else 4 END as apps_level_order,
a.apps_level_detailed,
CASE WHEN a.std_fwk_flag = 'Standard' THEN ifa.std_fwk_name ELSE a.fwk_name END AS std_fwk_name,
case when a.std_fwk_flag = 'Framework' then a.ssa_t1_code  when a.std_fwk_flag = 'Standard' then ssa.ssa_tier_1_code   else 'Error' end AS ssa_tier_1_code,
case when a.std_fwk_flag = 'Framework' then a.ssa_t1_desc  when a.std_fwk_flag = 'Standard' then ssa.ssa_tier_1  else 'Error' end AS ssa_t1_desc,
case when a.std_fwk_flag = 'Framework' then a.ssa_t2_code  when a.std_fwk_flag = 'Standard' then ssa.ssa_tier_2_code  else 'Error' end AS ssa_tier_2_code,
case when a.std_fwk_flag = 'Framework' then a.ssa_t2_desc  when a.std_fwk_flag = 'Standard' then ssa.ssa_tier_2  else 'Error' end AS ssa_t2_desc,
case when a.std_fwk_flag = 'Framework' then a.stem_apps when a.std_fwk_flag = 'Standard' and ssa.ssa_tier_1 in ('Science and Mathematics','Engineering and Manufacturing Technologies',
            'Construction, Planning and the Built Environment','Digital Technology') then 'STEM - Yes'   else 'STEM - No' end as stem,
std_fwk_flag,
provider_type,
--name as provider_name,
name_with_ukprn as provider_name,
learner_home_region,
CASE WHEN learner_home_region = 'North East' then 1
   WHEN learner_home_region = 'North West' then 2
	 WHEN learner_home_region = 'Yorkshire and The Humber' then 3
	 WHEN learner_home_region = 'East Midlands' then 4
	 WHEN learner_home_region = 'West Midlands' then 5
   WHEN learner_home_region = 'East of England' then 6
	 WHEN learner_home_region = 'London' then 7
	 WHEN learner_home_region = 'South East' then 8
	 WHEN learner_home_region = 'South West' then 9
	 WHEN learner_home_region = 'Outside of England and unknown' then 10
   ELSE 11 END as learner_home_region_order,
learner_home_la,
learner_home_lad,
learner_home_devolved_administration,
CASE WHEN learner_home_devolved_administration = 'Cambridgeshire and Peterborough' then 1
     WHEN learner_home_devolved_administration = 'Greater London Authority' then 2
	 WHEN learner_home_devolved_administration = 'Greater Manchester' then 3
	 WHEN learner_home_devolved_administration = 'Liverpool City Region' then 4
	 WHEN learner_home_devolved_administration = 'North East' then 5
	 WHEN learner_home_devolved_administration = 'North of Tyne' then 6
	 WHEN learner_home_devolved_administration = 'Sheffield City Region' then 7
	 WHEN learner_home_devolved_administration = 'South Yorkshire' then 8
	 WHEN learner_home_devolved_administration = 'Tees Valley' then 9
	 WHEN learner_home_devolved_administration = 'West Midlands' then 10
	 WHEN learner_home_devolved_administration = 'West of England' then 11
	 WHEN learner_home_devolved_administration = 'West Yorkshire' then 12
	 WHEN learner_home_devolved_administration = 'Outside of an English Devolved Area and unknown' then 13
ELSE 14 END as learner_home_devolved_administration_order,
delivery_region,
CASE WHEN delivery_region = 'North East' then 1
   WHEN delivery_region = 'North West' then 2
	 WHEN delivery_region = 'Yorkshire and The Humber' then 3
	 WHEN delivery_region = 'East Midlands' then 4
	 WHEN delivery_region = 'West Midlands' then 5
	 WHEN delivery_region = 'East of England' then 6
	 WHEN delivery_region = 'London' then 7
	 WHEN delivery_region = 'South East' then 8
	 WHEN delivery_region = 'South West' then 9
	 WHEN delivery_region = 'Outside of England and unknown' then 10
   ELSE 11 END as delivery_region_order,
delivery_la,
delivery_lad,
delivery_devolved_administration,
CASE WHEN delivery_devolved_administration = 'Cambridgeshire and Peterborough' then 1
   WHEN delivery_devolved_administration = 'Greater London Authority' then 2
	 WHEN delivery_devolved_administration = 'Greater Manchester' then 3
	 WHEN delivery_devolved_administration = 'Liverpool City Region' then 4
	 WHEN delivery_devolved_administration = 'North East' then 5
	 WHEN delivery_devolved_administration = 'North of Tyne' then 6
	 WHEN delivery_devolved_administration = 'Sheffield City Region' then 7
	 WHEN delivery_devolved_administration = 'South Yorkshire' then 8
	 WHEN delivery_devolved_administration = 'Tees Valley' then 9
	 WHEN delivery_devolved_administration = 'West Midlands' then 10
	 WHEN delivery_devolved_administration = 'West of England' then 11
	 WHEN delivery_devolved_administration = 'West Yorkshire' then 12
	 WHEN delivery_devolved_administration = 'Outside of an English Devolved Area and unknown' then 13
ELSE 14 END as delivery_devolved_administration_order,
starts_sr as starts,
achievements_sr as achievements,
enrolments_sr as enrolments

FROM catalog_40_copper_proj_fe_skills_statistics_dev.fe_skills_dev.vw_apprenticeship_start_ach_il_ees_all_years  a

left outer join catalog_40_copper_proj_fe_skills_statistics_dev.ref.routes_ifa ifa      on     ifa.std_lars_code = a.std_fwk_code and ifa.				academic_year = ${academic_year} and ifa.snapshot = ${snapshot}  and std_fwk_flag = 'Standard'

left join catalog_40_copper_proj_fe_skills_statistics_dev.ref.ssa_lookup ssa            on     ifa.ssa_tier_2_code_se = ssa.ssa_tier_2_code_se

where (a.academic_year = ${academic_year} and a.snapshot = ${snapshot}) or 
(a.academic_year in (${academic_year} - 101,${academic_year} - 202,${academic_year} - 303) and a.snapshot = 14)




In [0]:

--Calculate measures and group data, and format year*
--*ie place a /(solidus) after the first 4 characters, so that date appears as, for example, 2022/23 rather than 202223

CREATE OR REPLACE TEMPORARY VIEW ALL_FINAL AS

SELECT 
concat(substring(`year`,1,4) , '/' , substring(`year`,5,22)) as `year`,
age_group,
--age_group_order,
apps_Level,
--apps_level_order,
apps_level_detailed,
std_fwk_name,
ssa_t1_desc,
ssa_t2_desc,
std_fwk_flag,
provider_type,
provider_name,
learner_home_region,
--learner_home_region_order,
learner_home_la,
learner_home_lad,
learner_home_devolved_administration,
--learner_home_devolved_administration_order,
delivery_region,
--delivery_region_order,
delivery_la,
delivery_lad,
delivery_devolved_administration,
--delivery_devolved_administration_order,
sum(starts) as starts,
sum(achievements) as achievements,
sum(enrolments) as enrolments

FROM APPS 

group by
`year`,age_group,age_group_order,apps_Level,apps_level_order,apps_level_detailed,
std_fwk_name,ssa_t1_desc,ssa_t2_desc,std_fwk_flag,
provider_type,provider_name,
learner_home_region,learner_home_region_order,learner_home_la,learner_home_lad,
learner_home_devolved_administration,learner_home_devolved_administration_order,
delivery_region,delivery_region_order,delivery_la,delivery_lad,
delivery_devolved_administration,delivery_devolved_administration_order;

SELECT * FROM ALL_FINAL